# **Ejemplo completo de Clustering con DBSCAN**

**Usando el dataset Iris.**

**📦 1. Importar librerías necesarias**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# Estilo visual
sns.set(style='whitegrid')

**🌸 2. Cargar dataset real: IRIS**

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

# Creamos un DataFrame para explorarlo
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df.head()

**🔧 3. Estandarizar los datos**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

**🔍 4. Determinar un buen valor de Eps (Ɛ) con K-dist Plot**

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)
distances = np.sort(distances[:, 4])  # Tomamos la 5ta distancia más cercana

plt.figure(figsize=(8, 5))
plt.plot(distances)
plt.title('K-dist plot (para estimar eps en DBSCAN)')
plt.xlabel('Puntos ordenados')
plt.ylabel('Distancia al 5to vecino más cercano')
plt.grid(True)
plt.show()

**⚙️ 5. Aplicar DBSCAN con Eps estimado**

In [ ]:
# Puedes ajustar este valor si en la curva ves otro mejor
eps_val = 0.8 # asociado al codo de la curva en la gráfica anterior

dbscan = DBSCAN(eps=eps_val, min_samples= 5 ) # MinPts : min_samples = número de features + 1 → min_samples = 4 + 1 = 5
db_labels = dbscan.fit_predict(X_scaled)

# Cuántos clusters detectó (sin contar ruido -1)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print(f"DBSCAN detectó {n_clusters} clusters")

**📉 6. Visualizar con PCA**

In [ ]:
# se usa PCA para dismunuir su dimensionalidad de características de 4 a 2.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for label in np.unique(db_labels):
    if label == -1:
        plt.scatter(X_pca[db_labels == label, 0], X_pca[db_labels == label, 1],
                    c='gray', label='Ruido', alpha=0.6)
    else:
        plt.scatter(X_pca[db_labels == label, 0], X_pca[db_labels == label, 1],
                    label=f'Cluster {label}')

plt.title(f'Clusters detectados por DBSCAN (eps={eps_val})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True)
plt.show()

**🧪 7. Probar con una nueva flor (caso de prueba)**

In [ ]:
# Un nuevo punto con características parecidas a Iris Setosa
nueva_flor = np.array([[5.1, 3.5, 1.4, 0.2]])
nueva_flor_scaled = scaler.transform(nueva_flor)

# Distancia del nuevo punto a los centroides no aplica en DBSCAN
# Pero podemos ver si está dentro del eps de algún grupo

# Usamos el modelo para predecir si la flor es considerada "parte de algún grupo"
vecinos = dbscan.fit(X_scaled)  # ya está ajustado, pero repetimos para claridad
nueva_label = dbscan.fit_predict(np.vstack([X_scaled, nueva_flor_scaled]))[-1]

if nueva_label == -1:
    print("La nueva flor es considerada ruido (no pertenece a ningún cluster).")
else:
    print(f"La nueva flor pertenecería al Cluster {nueva_label}")

**📍 8. Visualizar la nueva flor en el gráfico PCA**

In [ ]:
# Volvemos a proyectar la nueva flor con PCA
nueva_flor_pca = pca.transform(nueva_flor_scaled)

plt.figure(figsize=(8, 6))
for label in np.unique(db_labels):
    if label == -1:
        plt.scatter(X_pca[db_labels == label, 0], X_pca[db_labels == label, 1],
                    c='gray', label='Ruido', alpha=0.5)
    else:
        plt.scatter(X_pca[db_labels == label, 0], X_pca[db_labels == label, 1],
                    label=f'Cluster {label}')

# Agregamos la nueva flor al gráfico
plt.scatter(nueva_flor_pca[0, 0], nueva_flor_pca[0, 1],
            c='red', marker='*', s=250, label='Nueva flor')

plt.title('Nueva flor proyectada sobre clusters (DBSCAN + PCA)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True)
plt.show()